In [20]:
import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import os

default_path = os.getcwd()
default_path

'/Users/avaldivia/Documents/GitHub/Split-Learning'

In [21]:
# Obtain the file path of the directory 'Split-Learning'
while not os.getcwd().endswith('Split-Learning'):
    os.chdir('..')
    if len(os.getcwd()) == 1:
        print("Unable to find the root directory")
        break
users_default_path = os.getcwd()
os.chdir(path = users_default_path)
users_default_path

'/Users/avaldivia/Documents/GitHub/Split-Learning'

In [22]:
files = [item for item in os.listdir() if item.endswith('csv.gz')]
#files.remove('CHARTEVENTS.csv.gz')
#files.remove('NOTEEVENTS.csv.gz')

# D_ICD_PROCEDURES

In [23]:
ICD_Procedures = (
    pl.scan_csv(source = 'D_ICD_PROCEDURES.csv.gz')
    .collect(streaming=True)
)
print(ICD_Procedures.shape)
ICD_Procedures.head(n = 2)

(3882, 4)


ROW_ID,ICD9_CODE,SHORT_TITLE,LONG_TITLE
i64,i64,str,str
264,851,"""Canthotomy""","""Canthotomy"""
265,852,"""Blepharorrhaphy""","""Blepharorrhaphy"""


In [24]:
ICD_Procedures_freq = ICD_Procedures['ICD9_CODE'].value_counts(sort=True)
ICD_Procedures_freq.filter(pl.col('count') > 2)

ICD9_CODE,count
i64,u32


In [25]:
ICD_Procedures['ROW_ID'].value_counts(sort=True).filter(pl.col('count') > 2)

ROW_ID,count
i64,u32


# D_ICD_DIAGNOSES

In [26]:
ICD_Diagnoses = (
    pl.scan_csv(source = 'D_ICD_DIAGNOSES.csv.gz', 
                schema_overrides = {'ICD9_CODE': pl.String})
    .collect(streaming=True)
)

print(ICD_Diagnoses.shape)
ICD_Diagnoses.head(n =2)

(14567, 4)


ROW_ID,ICD9_CODE,SHORT_TITLE,LONG_TITLE
i64,str,str,str
174,"""01166""","""TB pneumonia-oth test""","""Tuberculous pneumonia [any for…"
175,"""01170""","""TB pneumothorax-unspec""","""Tuberculous pneumothorax, unsp…"


# ADMISSIONS

In [27]:
admission = pl.read_csv('ADMISSIONS.csv.gz').sort(by = ['ROW_ID'])
#admission.sql(query="")
#admdission['ROW_ID'].value_counts().filter(pl.col('count') > 2) # Shows that every ROW_ID is unique
admission.head(n = 2)

ROW_ID,SUBJECT_ID,HADM_ID,ADMITTIME,DISCHTIME,DEATHTIME,ADMISSION_TYPE,ADMISSION_LOCATION,DISCHARGE_LOCATION,INSURANCE,LANGUAGE,RELIGION,MARITAL_STATUS,ETHNICITY,EDREGTIME,EDOUTTIME,DIAGNOSIS,HOSPITAL_EXPIRE_FLAG,HAS_CHARTEVENTS_DATA
i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64
1,2,163353,"""2138-07-17 19:04:00""","""2138-07-21 15:48:00""",null,"""NEWBORN""","""PHYS REFERRAL/NORMAL DELI""","""HOME""","""Private""",null,"""NOT SPECIFIED""",null,"""ASIAN""",null,null,"""NEWBORN""",0,1
2,3,145834,"""2101-10-20 19:08:00""","""2101-10-31 13:58:00""",null,"""EMERGENCY""","""EMERGENCY ROOM ADMIT""","""SNF""","""Medicare""",null,"""CATHOLIC""","""MARRIED""","""WHITE""","""2101-10-20 17:09:00""","""2101-10-20 19:24:00""","""HYPOTENSION""",0,1


In [28]:
#ICD_Procedures['ROW_ID'].value_counts(sort=True).filter(pl.col('count') > 2)
admission['SUBJECT_ID'].value_counts(sort=True).filter(pl.col('count') > 0)

SUBJECT_ID,count
i64,u32
13033,42
109,34
11861,34
5060,31
20643,24
…,…
99985,1
99991,1
99992,1


In [29]:
admission.select(pl.all().is_null().sum())

ROW_ID,SUBJECT_ID,HADM_ID,ADMITTIME,DISCHTIME,DEATHTIME,ADMISSION_TYPE,ADMISSION_LOCATION,DISCHARGE_LOCATION,INSURANCE,LANGUAGE,RELIGION,MARITAL_STATUS,ETHNICITY,EDREGTIME,EDOUTTIME,DIAGNOSIS,HOSPITAL_EXPIRE_FLAG,HAS_CHARTEVENTS_DATA
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,53122,0,0,0,0,25332,458,10128,0,28099,28099,25,0,0


# NOTE EVENTS

In [30]:
#notes = ()
# q1 = (
#     pl.scan_csv(source = 'NOTEEVENTS.csv.gz',low_memory = True)
#     .collect(streaming=True)
# )

In [31]:
#print(q1['TEXT'][1])

In [32]:
#print(q1['TEXT'][0])

# JOIN datasets

In [33]:
admission.head(n=1)

ROW_ID,SUBJECT_ID,HADM_ID,ADMITTIME,DISCHTIME,DEATHTIME,ADMISSION_TYPE,ADMISSION_LOCATION,DISCHARGE_LOCATION,INSURANCE,LANGUAGE,RELIGION,MARITAL_STATUS,ETHNICITY,EDREGTIME,EDOUTTIME,DIAGNOSIS,HOSPITAL_EXPIRE_FLAG,HAS_CHARTEVENTS_DATA
i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64
1,2,163353,"""2138-07-17 19:04:00""","""2138-07-21 15:48:00""",null,"""NEWBORN""","""PHYS REFERRAL/NORMAL DELI""","""HOME""","""Private""",null,"""NOT SPECIFIED""",null,"""ASIAN""",null,null,"""NEWBORN""",0,1


In [34]:
ICD_Procedures.head(n = 1)

ROW_ID,ICD9_CODE,SHORT_TITLE,LONG_TITLE
i64,i64,str,str
264,851,"""Canthotomy""","""Canthotomy"""


In [35]:
ICD_Diagnoses.head(n=1)

ROW_ID,ICD9_CODE,SHORT_TITLE,LONG_TITLE
i64,str,str,str
174,"""01166""","""TB pneumonia-oth test""","""Tuberculous pneumonia [any for…"


In [39]:
print(min(admission['ROW_ID']), max(admission['ROW_ID']))
print(min(ICD_Procedures['ROW_ID']), max(ICD_Procedures['ROW_ID']) )
print(min(ICD_Diagnoses['ROW_ID']), max(ICD_Diagnoses['ROW_ID']))

1 58976
1 3882
1 14567


In [36]:
print(admission.shape)
print(ICD_Procedures.shape)
print(ICD_Diagnoses.shape)

(58976, 19)
(3882, 4)
(14567, 4)


In [37]:
dataset = admission.join(other = ICD_Procedures , on = 'ROW_ID', how = 'inner')
dataset = dataset  .join(other = ICD_Diagnoses  , on = 'ROW_ID', how = 'inner')
dataset.head()

ROW_ID,SUBJECT_ID,HADM_ID,ADMITTIME,DISCHTIME,DEATHTIME,ADMISSION_TYPE,ADMISSION_LOCATION,DISCHARGE_LOCATION,INSURANCE,LANGUAGE,RELIGION,MARITAL_STATUS,ETHNICITY,EDREGTIME,EDOUTTIME,DIAGNOSIS,HOSPITAL_EXPIRE_FLAG,HAS_CHARTEVENTS_DATA,ICD9_CODE,SHORT_TITLE,LONG_TITLE,ICD9_CODE_right,SHORT_TITLE_right,LONG_TITLE_right
i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,str,str,str,str,str
174,129,164174,"""2157-03-27 08:46:00""","""2157-04-05 14:00:00""",null,"""EMERGENCY""","""EMERGENCY ROOM ADMIT""","""REHAB/DISTINCT PART HOSP""","""Government""","""SPAN""","""CATHOLIC""","""MARRIED""","""HISPANIC/LATINO - GUATEMALAN""","""2157-03-27 03:58:00""","""2157-03-27 10:21:00""","""PARENCHYMAL HEMORRHAGE""",0,1,50,"""Impl CRT pacemaker sys""","""Implantation of cardiac resync…","""01166""","""TB pneumonia-oth test""","""Tuberculous pneumonia [any for…"
175,130,198214,"""2119-10-29 14:49:00""","""2119-11-05 17:00:00""",null,"""EMERGENCY""","""EMERGENCY ROOM ADMIT""","""HOME HEALTH CARE""","""Private""",null,"""CATHOLIC""","""SINGLE""","""WHITE""","""2119-10-29 12:02:00""","""2119-10-29 17:00:00""","""RULE-OUT MYOCARDIAL INFARCTION…",0,1,51,"""Impl CRT defibrillat sys""","""Implantation of cardiac resync…","""01170""","""TB pneumothorax-unspec""","""Tuberculous pneumothorax, unsp…"
176,130,113323,"""2119-11-14 12:00:00""","""2119-12-02 11:40:00""",null,"""EMERGENCY""","""PHYS REFERRAL/NORMAL DELI""","""SNF""","""Private""",null,"""CATHOLIC""","""SINGLE""","""WHITE""",null,null,"""STERNAL DRAINAGE;SHORTNESS OF …",0,1,52,"""Imp/rep lead lf ven sys""","""Implantation or replacement of…","""01171""","""TB pneumothorax-no exam""","""Tuberculous pneumothorax, bact…"
177,131,171781,"""2143-12-07 12:19:00""","""2143-12-12 09:00:00""",null,"""NEWBORN""","""CLINIC REFERRAL/PREMATURE""","""HOME HEALTH CARE""","""Medicaid""",null,"""NOT SPECIFIED""",null,"""BLACK/AFRICAN AMERICAN""",null,null,"""NEWBORN""",0,1,53,"""Imp/rep CRT pacemakr gen""","""Implantation or replacement of…","""01172""","""TB pneumothorx-exam unkn""","""Tuberculous pneumothorax, bact…"
178,132,160192,"""2115-05-06 21:53:00""","""2115-05-25 12:10:00""",null,"""EMERGENCY""","""EMERGENCY ROOM ADMIT""","""REHAB/DISTINCT PART HOSP""","""Private""",null,"""CATHOLIC""","""WIDOWED""","""WHITE""","""2115-05-06 12:18:00""","""2115-05-06 14:00:00""","""SUBARACHNOID HEMORRHAGE""",0,1,54,"""Imp/rep CRT defib genat""","""Implantation or replacement of…","""01173""","""TB pneumothorax-micro dx""","""Tuberculous pneumothorax, tube…"


In [ ]:
df_g = dataset.group_by('SUBJECT_ID').agg(
    pl.col('ICD9_CODE').str.join(',')
)

In [ ]:
i = 20
df_g['ICD9_CODE'][i:i + 20]

In [ ]:
import pandas as pd

In [ ]:
dataset['ICD9_CODE_right'].value_counts(sort=True).head()

In [ ]:
dataset['ICD9_CODE'].value_counts(sort=True).head()

In [ ]:
diagnosis_labels = ['4019', '4280', '41401', '42731', '25000', '5849', '2724', '51881', '53081', '5990', '2720',
                    '2859', '2449', '486', '2762', '2851', '496', 'V5861', '99592', '311', '0389', '5859', '5070',
                    '40390', '3051', '412', 'V4581', '2761', '41071', '2875', '4240', 'V1582', 'V4582', 'V5867',
                    '4241', '40391', '78552', '5119', '42789', '32723', '49390', '9971', '2767', '2760', '2749',
                    '4168', '5180', '45829', '4589', '73300', '5845', '78039', '5856', '4271', '4254', '4111',
                    'V1251', '30000', '3572', '60000', '27800', '41400', '2768', '4439', '27651', 'V4501', '27652',
                    '99811', '431', '28521', '2930', '7907', 'E8798', '5789', '79902', 'V4986', 'V103', '42832',
                    'E8788', '00845', '5715', '99591', '07054', '42833', '4275', '49121', 'V1046', '2948', '70703',
                    '2809', '5712', '27801', '42732', '99812', '4139', '3004', '2639', '42822', '25060', 'V1254',
                    '42823', '28529', 'E8782', '30500', '78791', '78551', 'E8889', '78820', '34590', '2800', '99859',
                    'V667', 'E8497', '79092', '5723', '3485', '5601', '25040', '570', '71590', '2869', '2763', '5770',
                    'V5865', '99662', '28860', '36201', '56210']
print(len(diagnosis_labels))

In [ ]:
dataset['ICD9_CODE'].value_counts(sort=True)[:50]['ICD9_CODE']

In [ ]:
dataset['ICD9_CODE_right'].value_counts(sort = True)

In [ ]:
dataset['ICD9_CODE_right'].value_counts().filter(pl.col('count') > 0)